# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedsamymohamad/flyrank_internship_starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring**

Four things in order:
1. The contract — five plain-words answers
2. Three verification queries with visible outputs (on `month=2026-03`)
3. Five features with "knowable when?" lines, plus the deliberate-leak experiment
4. One named limitation

> ⚠️ **Before running:** request gate access at https://huggingface.co/datasets/FlyRank/internship-warehouse (instant approval), create a plain **Read** token in your HF settings, and store it as a Colab Secret named `HF_TOKEN`. Never paste the token into a cell — this repo is public.

---
## 0. Setup — install, authenticate, connect

In [ ]:
# Install dependencies (Colab already has most; the pip call is fast)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'huggingface_hub', 'duckdb>=0.10'], check=True)
print('Dependencies ready.')

In [ ]:
import os

# Retrieve the token from Colab Secrets (key icon in the left panel)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    # Fallback for local execution: read from environment
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise EnvironmentError(
        'HF_TOKEN not found. Add it as a Colab Secret named HF_TOKEN '
        '(the key panel on the left). Never paste the token into a cell.'
    )

# Authenticate once — needed for the gated dataset
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print('Authenticated with Hugging Face.')

In [ ]:
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET force_download=false;")
con.execute(f"SET hf_token='{HF_TOKEN}';")

# Mid-panel month — safe for developing label logic
MONTH = 'month=2026-03'

FACT_PATH = (
    f"hf://datasets/FlyRank/internship-warehouse/"
    f"fact_content_daily_performance/{MONTH}/*.parquet"
)

# Smoke-test: just count rows in this partition
n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{FACT_PATH}')").fetchone()[0]
print(f'Partition {MONTH}: {n:,} rows — connection OK.')

---
## 1. The Contract — five plain-words answers

### 1.1 — One row means…

One row = **one content item (page) on one calendar date for one client**.
The grain is `(report_date, client_id, content_id)` — a single day's measured performance for a single page belonging to a single client.

### 1.2 — Table(s) I'll use

**Primary:** `fact_content_daily_performance`, partitioned by `month=YYYY-MM`. I develop on `month=2026-03` (a mid-panel month). I treat `month=2026-06` (the `_sample` table) as a **sealed test month** — it is the natural outcome window of any past→future label, so label logic must never be developed there.

**Secondary join (if needed):** `dim_content` for static content metadata (word count, content type, content age). Joined on `content_id`.

### 1.3 — Time window

The feature window is the **30 days before the label period** (`impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` — days 31–60 before snapshot). The label window is the **most-recent 30 days** (`impressions_last_30d` vs `impressions_prev_30d`). I do not use any column that overlaps with the label window as a feature — `impressions_last_30d` and `clicks_last_30d` are off-limits.

### 1.4 — What I predict (label / proxy)

**Proxy label:** `is_declining = 1` when `impressions_last_30d` fell more than 20% compared with `impressions_prev_30d` (i.e. the same rule as `trend_direction == "down"` in the starter CSV). This is a **rule-based proxy**, not a directly observed outcome — I state that clearly when reporting results.

### 1.5 — One deliberate exclusion

I exclude **`impressions_last_30d`** and any column derived from it (`trend_direction`, `trend_pct` in the starter CSV; any column that is a function of the last-30d window in the warehouse). These columns partially or fully encode the label. Using them would be label leakage — a model trained with them would appear to learn but is simply reading the answer. Section 3 demonstrates this deliberately.

In [ ]:
# Print the contract answers as a quick reference card
contract = {
    '1 — One row':    'one content item on one report_date for one client '
                      '→ grain: (report_date, client_id, content_id)',
    '2 — Table':      'fact_content_daily_performance, partition month=2026-03 '
                      '(+ dim_content for static metadata if needed)',
    '3 — Window':     'features from prev_30d window (days 31-60 back); '
                      'label from last_30d vs prev_30d comparison',
    '4 — Label':      'is_declining = 1 when impressions_last_30d/impressions_prev_30d < 0.80 '
                      '(rule-based proxy; not a directly observed outcome)',
    '5 — Excluded':   'impressions_last_30d, clicks_last_30d, sessions_last_30d '
                      '(overlap label window → leakage); trend_direction, trend_pct '
                      '(derived from label); content_id, client_id (IDs, not features)',
}

print('=== DATA CONTRACT — Lane 2: Refresh / Content Opportunity Scoring ===')
for k, v in contract.items():
    print(f'\n{k}:\n  {v}')

---
## 2. Verify it — three queries with visible outputs

All three queries run on `month=2026-03`. A claim without a query under it is a guess.

In [ ]:
# ── Query 1: GRAIN ───────────────────────────────────────────────────────────
# If the grain claim holds, no (report_date, client_id, content_id) triple
# appears more than once. Zero rows back = grain is clean.

q1 = f"""
SELECT
    report_date,
    client_id,
    content_id,
    COUNT(*) AS c
FROM read_parquet('{FACT_PATH}')
GROUP BY report_date, client_id, content_id
HAVING COUNT(*) > 1
LIMIT 5
"""

import pandas as pd
result_q1 = con.execute(q1).df()

print('Query 1 — Grain check: rows where (report_date, client_id, content_id) appears > 1 time')
print(f'Rows returned: {len(result_q1)}  (0 = grain holds ✓)')
print(result_q1)

In [ ]:
# ── Query 2: ROW COUNT + DATE SPAN ────────────────────────────────────────────
# Confirm how many rows are in this partition and the exact date range.

q2 = f"""
SELECT
    COUNT(*)           AS total_rows,
    COUNT(DISTINCT client_id)  AS unique_clients,
    COUNT(DISTINCT content_id) AS unique_pages,
    MIN(report_date)   AS earliest_date,
    MAX(report_date)   AS latest_date
FROM read_parquet('{FACT_PATH}')
"""

result_q2 = con.execute(q2).df()

print('Query 2 — Row count and date span for month=2026-03')
print(result_q2.T.to_string())

In [ ]:
# ── Query 3: AVAILABILITY — filter with IS TRUE ───────────────────────────────
# The flag ga4_data_available is three-valued: TRUE, FALSE, or NULL.
# Rows where it is NULL have null metrics — neither zero-filled nor flagged FALSE.
# We must use IS TRUE (not = TRUE) to handle NULLs correctly.
#
# Claim: after filtering ga4_data_available IS TRUE, we retain a meaningful
# subset of rows that have real GA4 engagement data.

q3 = f"""
SELECT
    COUNT(*)                                                      AS total_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)            AS ga4_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)
        * 100.0 / COUNT(*)                                        AS ga4_available_pct,
    COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE)        AS ga4_unavailable_or_null_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)            AS gsc_available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)
        * 100.0 / COUNT(*)                                        AS gsc_available_pct
FROM read_parquet('{FACT_PATH}')
"""

result_q3 = con.execute(q3).df()

print('Query 3 — Availability: IS TRUE filter on ga4_data_available and gsc_data_available')
print(result_q3.T.to_string())
print()
print('Note: IS TRUE (not = TRUE) correctly excludes NULL rows, which carry null metrics.')
print('Only ga4_available_rows have real engagement signals safe to use as features.')

---
## 3. Five features + the leakage trap

### 3a. Feature frame

All five features are from the **prev_30d window** (days 31–60 before snapshot) or from static content properties. None overlaps with the label window (last_30d).

| Feature | Column(s) | Available when? |
|---|---|---|
| `log_impressions_prev30` | `log1p(impressions_prev_30d)` | Knowable at decision moment because it covers days 31–60 before the snapshot — entirely before the label's last-30d window |
| `log_clicks_prev30` | `log1p(clicks_prev_30d)` | Knowable at decision moment — same prev_30d window, no overlap with label |
| `avg_position` | `gsc_avg_position` | Knowable at decision moment — the average GSC position over the trailing 90-day aggregate; not derived from any last_30d column |
| `days_with_impressions` | `days_with_impressions` | Knowable at decision moment — count of days in the 90-day window that had ≥1 impression; measures visibility regularity, not the trend direction |
| `log_sessions_prev30` | `log1p(sessions_prev_30d)` | Knowable at decision moment — GA4 sessions in days 31–60 back; safe only after filtering `ga4_data_available IS TRUE` |

In [ ]:
import numpy as np
import pandas as pd

# Build the feature frame from month=2026-03
# We restrict to rows where both GSC and GA4 data are available (IS TRUE)
# so that all five features carry real signal (not zero-filled nulls).

q_features = f"""
SELECT
    report_date,
    client_id,
    content_id,

    -- Five safe features (all from prev_30d or 90d aggregates)
    LN(1 + impressions_prev_30d)  AS log_impressions_prev30,
    LN(1 + clicks_prev_30d)       AS log_clicks_prev30,
    gsc_avg_position               AS avg_position,
    days_with_impressions          AS days_with_impressions,
    LN(1 + sessions_prev_30d)     AS log_sessions_prev30,

    -- Label (rule-based proxy: declining = last_30d dropped >20%)
    CASE
        WHEN impressions_prev_30d > 0
             AND impressions_last_30d < impressions_prev_30d * 0.80
        THEN 1 ELSE 0
    END AS is_declining

FROM read_parquet('{FACT_PATH}')
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
  AND impressions_prev_30d > 0   -- need a baseline to compute the label
LIMIT 50000                      -- cap for development; full scan runs once query is final
"""

feat = con.execute(q_features).df()

print(f'Feature frame shape: {feat.shape}')
print(f'Label rate (is_declining = 1): {feat["is_declining"].mean():.1%}')
print()
print('First 5 rows of the feature frame:')
feat.head()

In [ ]:
# Summary statistics for the five features
feature_cols = [
    'log_impressions_prev30', 'log_clicks_prev30',
    'avg_position', 'days_with_impressions', 'log_sessions_prev30'
]
print('Feature summary statistics:')
feat[feature_cols].describe().round(3)

### 3b. The leakage trap — watch the score jump, then delete the leak

Here we deliberately add `impressions_last_30d` as a feature — the numerator of the declining label — to watch what happens to the model score. Then we remove it and report the honest number.

> **The lesson from notebook 02, performed on real warehouse data:** a column that encodes (or is derived from) the label will make the model look nearly perfect. That performance is not real — the model is reading the answer, not learning from signals. Any column that overlaps with the label's computation window must be excluded before a single line of model code runs.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# ── Pull the LEAKED frame (adds impressions_last_30d as a 6th feature) ────
q_leaked = f"""
SELECT
    LN(1 + impressions_prev_30d)  AS log_impressions_prev30,
    LN(1 + clicks_prev_30d)       AS log_clicks_prev30,
    gsc_avg_position               AS avg_position,
    days_with_impressions          AS days_with_impressions,
    LN(1 + sessions_prev_30d)     AS log_sessions_prev30,

    -- THE LEAK: last_30d impressions — partially encodes the label
    LN(1 + impressions_last_30d)  AS log_impressions_last30_LEAK,

    CASE
        WHEN impressions_prev_30d > 0
             AND impressions_last_30d < impressions_prev_30d * 0.80
        THEN 1 ELSE 0
    END AS is_declining

FROM read_parquet('{FACT_PATH}')
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
  AND impressions_prev_30d > 0
LIMIT 50000
"""

feat_leaked = con.execute(q_leaked).df().dropna()
y = feat_leaked['is_declining']

clf = Pipeline([('scaler', StandardScaler()),
                ('lr', LogisticRegression(max_iter=1000))])

# Score WITH the leak
X_leaked = feat_leaked[[
    'log_impressions_prev30', 'log_clicks_prev30', 'avg_position',
    'days_with_impressions', 'log_sessions_prev30',
    'log_impressions_last30_LEAK'   # <-- the leaker
]]
auc_leaked = cross_val_score(clf, X_leaked, y, cv=5, scoring='roc_auc').mean()
print(f'[LEAKED  ] ROC-AUC with impressions_last_30d included : {auc_leaked:.4f}  ← suspiciously high')

# Score WITHOUT the leak (honest)
X_honest = feat_leaked[[
    'log_impressions_prev30', 'log_clicks_prev30', 'avg_position',
    'days_with_impressions', 'log_sessions_prev30'
]]
auc_honest = cross_val_score(clf, X_honest, y, cv=5, scoring='roc_auc').mean()
print(f'[HONEST  ] ROC-AUC after removing the leak             : {auc_honest:.4f}  ← real signal')
print()
print(f'AUC gap caused by leakage: {auc_leaked - auc_honest:+.4f}')
print()
print('Conclusion: impressions_last_30d is DELETED from the feature set.')
print('The honest ROC-AUC above is the number we carry forward.')

In [ ]:
# Confirm: the final feature set contains NO label-window columns
final_features = [
    'log_impressions_prev30',
    'log_clicks_prev30',
    'avg_position',
    'days_with_impressions',
    'log_sessions_prev30',
]
excluded = [
    'impressions_last_30d',
    'clicks_last_30d',
    'sessions_last_30d',
    'trend_direction',
    'trend_pct',
]

print('Final feature set (no label-window columns):')
for f in final_features:
    print(f'  ✓  {f}')
print()
print('Excluded (would be leakage):')
for e in excluded:
    print(f'  ✗  {e}')

---
## 4. Data limits — one named limitation

**Unbalanced panel depth makes cross-client normalisation necessary.**

The `fact_content_daily_performance` panel is unbalanced: per-client history depth ranges from roughly 3 to 17 months depending on each client's `gsc_data_start` (stored in `dim_clients`). In `month=2026-03`, clients who joined the platform after mid-2025 may have fewer than 60 days of prior-period history — meaning their `impressions_prev_30d` values reflect a shorter (or zero) baseline rather than a true 30-day trailing window. A model trained on raw cross-client values will see artificially low `impressions_prev_30d` for new clients and could conflate "new client with limited history" with "declining page". Any cross-client model must either (a) normalise features per client, (b) use a per-client baseline ratio, or (c) filter to clients with `gsc_data_start ≤ 2025-08` before computing the label. This limitation is not visible in the per-row data; it requires a join to `dim_clients`.

In [ ]:
# Illustrate the panel imbalance: count rows per client in month=2026-03
q_panel = f"""
SELECT
    client_id,
    COUNT(*)          AS row_count,
    MIN(report_date)  AS first_date,
    MAX(report_date)  AS last_date,
    COUNT(DISTINCT report_date) AS distinct_days
FROM read_parquet('{FACT_PATH}')
WHERE gsc_data_available IS TRUE
GROUP BY client_id
ORDER BY row_count ASC
LIMIT 10
"""

result_panel = con.execute(q_panel).df()
print('Ten smallest clients by row count in month=2026-03 (bottom of the panel):')
print(result_panel.to_string(index=False))
print()
print('Limitation confirmed: history depth varies per client — '
      'a per-client normalisation is needed before cross-client modelling.')

---
## 5. Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Three verification queries with visible outputs — grain, counts/dates, IS TRUE availability
- [x] Five features, each with an "available when?" line
- [x] The deliberate-leak experiment is shown, quantified, then removed — honest number kept
- [x] One named limitation stated in plain words and illustrated with a query
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.